# 01 — Coastal Virginia elevation (retargeted)

High-ground search for the relocation: a house in southeastern Virginia within
Larissa's **30-mile-of-the-beach** rule, with the surf/water access handled
separately by a marina slip in the northern OBX. Because the house no longer
carries the water-access job, it's free to sit on inland high ground — which is
exactly where the safe ground is here.

Retargeted from the 17-county NC pipeline. What carried over unchanged: the
20 ft NAVD88 threshold, the bare-earth DTM requirement, the buildable-pad test,
the cell-size assertion, 10 m 3DEP resolution, UTM 18N. What changed: the area
(three VA localities), and one hard gate that didn't exist in NC — the beach
rule.

## The landscape, in one paragraph

This is not Carteret. There are no waterfront bluffs. The relief here is the
**Suffolk Scarp**, a relict shoreline ~20 mi inland: ground east of it is the
low terrace that stood underwater at a +20-25 ft highstand, ground on and west
of it is the buildable high ground. Larissa's rule pulls the search *east*
(toward the beach, low and wet); the 20 ft rule pulls it *west* (inland, high
and dry). The winners are the thin band where the scarp's eastern reach still
falls inside 30 miles. **Expect far fewer hits than NC** — that's the terrain
being honest, not the pipeline failing.

## One caveat this coast adds: subsidence

Hampton Roads has among the highest *relative* sea-level-rise rates in the
country because the land is actively sinking. The 10 m DEM measures today's
ground; a finalist's true long-horizon margin is a touch less than measured.
Reason enough to keep the 20 ft threshold and the pond-spoil pad-raise, and to
end on 1 m lidar before any offer.

## 1 · Config

In [2]:
from pathlib import Path
import numpy as np, pandas as pd
import warnings; warnings.filterwarnings("ignore")

import py3dep, pygris
import geopandas as gpd
import rasterio
from rasterio.transform import from_origin
from shapely.geometry import mapping
from scipy import ndimage

import va_aoi as A          # shared geography (localities, beach rule, exclusions)
import nc_palette as pal    # unchanged banded palette

DEM_DIR = Path("data/dem");     DEM_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR = Path("output");       OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "maps").mkdir(exist_ok=True)

M_TO_FT = 3.280839895
DEM_RES = 10                    # 3DEP 1/3 arc-second, ~10 m

print("localities:", ", ".join(A.LOCALITIES))
print(f"beach rule: {A.BEACH_RULE_MI:g} mi  |  coarse buffer: {A.BEACH_BUFFER_MI:g} mi")
print(f"threshold:  {A.THRESHOLD_FT:g} ft NAVD88")

localities: Virginia_Beach, Chesapeake, Suffolk
beach rule: 30 mi  |  coarse buffer: 31 mi
threshold:  20 ft NAVD88


## 2 · Build the clipped AOI

In [3]:
# --- Build the clipped AOI --------------------------------------------------
# Locality polygons intersected with the beach buffer. We fetch DEM only for the
# in-range slice -- no point pulling western Suffolk we'll never use.

cty = pygris.counties(state=A.STATE_FIPS, cb=True, year=2023, cache=True)
cty["GEOID"] = cty["GEOID"].astype(str)
targets = cty[cty.GEOID.isin(A.LOCALITIES.values())].to_crs(A.WORKING_EPSG)
print(f"{len(targets)} localities pulled:", ", ".join(sorted(targets.NAME)))

buffer_utm = A.beach_buffer_geom()          # UTM 18N meters
targets["clipped"] = targets.geometry.intersection(buffer_utm)
targets = targets[~targets.clipped.is_empty].copy()

for _, r in targets.iterrows():
    full = r.geometry.area / 1609.344**2
    clip = r.clipped.area / 1609.344**2
    print(f"  {r.NAME:<16} {clip:6.1f} / {full:6.1f} sq mi in range "
          f"({100*clip/full:4.0f}%)")

3 localities pulled: Chesapeake, Suffolk, Virginia Beach
  Suffolk            79.1 /  416.3 sq mi in range (  19%)
  Virginia Beach    306.5 /  306.5 sq mi in range ( 100%)
  Chesapeake        345.7 /  350.7 sq mi in range (  99%)


## 3 · Fetch 3DEP per locality

In [4]:
import time
import numpy as np
from shapely.geometry import box
from rioxarray.merge import merge_arrays

def fetch_dem_tiled(name, geom_utm, tile_deg=0.07, res=DEM_RES, tries=4):
    """Fetch one locality as small tiles and mosaic — reliable when whole-locality fails."""
    fp = DEM_DIR / f"{name}.tif"
    if fp.exists():
        print(f"  cached  {name}")
        return fp
    geom_wgs = gpd.GeoSeries([geom_utm], crs=A.WORKING_EPSG).to_crs(4326).iloc[0]
    minx, miny, maxx, maxy = geom_wgs.bounds
    nx = int(np.ceil((maxx - minx) / tile_deg))
    ny = int(np.ceil((maxy - miny) / tile_deg))
    pad = 0.0015
    print(f"  {name}: {nx}x{ny} grid")
    tiles = []
    for i in range(nx):
        for j in range(ny):
            cell = box(minx + i*tile_deg - pad, miny + j*tile_deg - pad,
                       minx + (i+1)*tile_deg + pad, miny + (j+1)*tile_deg + pad)
            piece = cell.intersection(geom_wgs)
            if piece.is_empty:
                continue
            for attempt in range(1, tries + 1):
                try:
                    tiles.append(py3dep.get_map("DEM", piece, resolution=res,
                                                geo_crs=4326, crs=4326))
                    break
                except Exception:
                    if attempt < tries:
                        time.sleep(3 * attempt)
            time.sleep(0.3)
    if not tiles:
        print(f"  [skip] {name}")
        return None
    mosaic = merge_arrays(tiles) if len(tiles) > 1 else tiles[0]
    mosaic = mosaic.rio.reproject(A.WORKING_EPSG)
    mosaic.rio.to_raster(fp)
    print(f"  [ok] {name}: {len(tiles)} tiles -> {mosaic.shape}")
    return fp

# Loop ALL three localities. Cached ones skip instantly; anything missing
# gets tiled automatically.
tifs = {}
for _, r in targets.iterrows():
    key = r.NAME.replace(" ", "_")
    fp = fetch_dem_tiled(key, r.geometry)
    if fp is not None:
        tifs[key] = fp
print(f"\n{len(tifs)} DEMs ready")

  cached  Suffolk
  cached  Virginia_Beach
  cached  Chesapeake

3 DEMs ready


## 4 · Find high-ground regions

In [ ]:
# ============================================================================
# 4 · Find high-ground regions
# ============================================================================
# For each locality: threshold the DEM at 20 ft -> label contiguous high-ground
# blobs -> buildable-pad test (largest inscribed circle, which catches
# unbuildable ribbon terraces) -> collect into a regions table. Beach distance
# is added later in triage; nothing here is gated, only measured.

# --- Elevation sanity ceiling ------------------------------------------------
# No bare ground in these three coastal localities exceeds ~40 ft. Anything
# above this is NOT terrain -- it's building/bridge surface returns or
# mosaic-edge fill from the reproject. Capping to NaN removes the
# rooftop-as-hill phantom regions AND the black speckle outside the county line.
# WARNING: coastal assumption. If the AOI ever extends inland (Isle of Wight,
# the Surry Scarp) where real ground tops 50 ft, RAISE this or it will delete
# legitimate high ground.
MAX_REAL_FT = 50.0


def largest_pad_acres(mask, cell_m):
    """Largest inscribed circle inside a region, in acres -- the buildable core."""
    if mask.sum() == 0:
        return 0.0
    edt = ndimage.distance_transform_edt(mask) * cell_m       # meters to edge
    return np.pi * edt.max() ** 2 / 4046.8564224


def analyze(name, path):
    """Read one locality's DEM, clean it, return (elevation_ft_array, regions_df)."""
    with rasterio.open(path) as src:
        z = src.read(1).astype("float32")
        assert src.crs.to_epsg() == A.WORKING_EPSG, f"{name}: wrong CRS"
        cw = abs(src.transform.a)
        # 3DEP serves 8-12 m depending on source lidar; accept that range, but
        # still scream if a DEM comes back in degrees (~0.0001) or wrong product.
        assert 5.0 <= cw <= 15.0, f"{name}: cell {cw:.1f} m -- expected ~{DEM_RES} m"
        nod = src.nodata
        tr = src.transform

    # --- clean the elevation array ---
    if nod is not None:
        z[z == nod] = np.nan
    z[z < -100] = np.nan                       # extreme sentinel junk
    ft = z * M_TO_FT                            # meters (NAVD88) -> feet
    ft[ft < -2] = np.nan                       # open water / nodata leak (low side)
    ft[ft > MAX_REAL_FT] = np.nan              # clutter & edge fill (high side)

    # --- label contiguous high-ground blobs ---
    high = np.nan_to_num(ft, nan=-999) >= A.THRESHOLD_FT
    lbl, n = ndimage.label(high)

    rows = []
    for i in range(1, n + 1):
        m = lbl == i
        acres = m.sum() * cw * cw / 4046.8564224   # uses THIS file's cell size
        if acres < 1.0:                            # drop specks
            continue
        ys, xs = np.where(m)
        cx, cy = tr * (xs.mean(), ys.mean())       # centroid -> map coords
        pad = largest_pad_acres(m, cw)
        rows.append(dict(
            locality=name, region_id=i, acres=acres,
            mean_ft=float(np.nanmean(ft[m])), max_ft=float(np.nanmax(ft[m])),
            pad_acres=pad, pad_fits=pad >= A.MIN_PAD_ACRES,
            x=cx, y=cy,
        ))
    return ft, pd.DataFrame(rows)


# --- run every locality, render its map, collect regions --------------------
all_regions, stats = [], []
for name, path in tifs.items():
    ft, df = analyze(name, path)

    fig_ax = pal.plot_county(ft, name.replace("_", " "))
    fig_ax.figure.savefig(OUT_DIR / "maps" / f"{name}.png",
                          dpi=130, bbox_inches="tight", facecolor="white")
    fig_ax.figure.clf()

    land = ~np.isnan(ft)
    stats.append(dict(
        locality=name,
        pct_over_20=100 * float((ft[land] >= A.THRESHOLD_FT).mean()),
        mean_ft=float(np.nanmean(ft)),
        max_ft=float(np.nanmax(ft)),
        n_regions=len(df),
        n_buildable=int(df.pad_fits.sum()) if len(df) else 0,
    ))
    all_regions.append(df)
    print(f"  {name:<16} {len(df):4d} regions, "
          f"{int(df.pad_fits.sum()) if len(df) else 0:3d} buildable, "
          f"{stats[-1]['pct_over_20']:5.1f}% >=20ft")

regions = pd.concat(all_regions, ignore_index=True) if all_regions else pd.DataFrame()
summary = pd.DataFrame(stats).sort_values("pct_over_20", ascending=False)
print()
print(summary.to_string(index=False, float_format=lambda v: f"{v:.2f}"))

## 5 · Beach distance + save

In [ ]:
# --- Attach beach distance, save --------------------------------------------
# Convert region centroids to lon/lat, add the coarse straight-line beach
# distance as a COLUMN. The gate happens in triage (02), not here.

if len(regions):
    ll = gpd.GeoSeries(gpd.points_from_xy(regions.x, regions.y),
                       crs=A.WORKING_EPSG).to_crs(4326)
    regions["lon"] = ll.x.values
    regions["lat"] = ll.y.values
    regions["beach_mi"] = A.dist_to_beach_mi(regions.lon.values, regions.lat.values)

regions.to_csv(OUT_DIR / "regions_20ft.csv", index=False)
summary.to_csv(OUT_DIR / "county_summary.csv", index=False)

print(summary.to_string(index=False, float_format=lambda v: f"{v:7.2f}"))
print(f"\n[ok] {len(regions)} regions -> output/regions_20ft.csv")
print(f"[ok] {len(tifs)} maps -> output/maps/")
if len(regions):
    within = (regions.beach_mi <= A.BEACH_RULE_MI).sum()
    print(f"\n{within}/{len(regions)} regions inside the {A.BEACH_RULE_MI:g}-mi "
          f"straight-line beach rule (drive-time refines these in triage)")

      locality  pct_over_20  mean_ft  max_ft  n_regions  n_buildable
       Suffolk        55.33    17.07   50.00        163           30
    Chesapeake         8.21    13.11   50.00        570           58
Virginia_Beach         4.26     7.17   50.00        328           40

[ok] 1061 regions -> output/regions_20ft.csv
[ok] 3 maps -> output/maps/

1029/1061 regions inside the 30-mi straight-line beach rule (drive-time refines these in triage)


In [ ]:
import os
for f in ["banded_Brunswick.png", "banded_Craven.png", "banded_New_Hanover.png",
          "banded_Pender.png", "threshold_sweep.png"]:
    p = OUT_DIR / f
    if p.exists():
        os.remove(p); print("removed", f)